# CashLog - Designing the optimal Cash-Center Network



# The problem

CashLog is the Spanish market leader in cash logistics. Concretely it does three things: 
- collect cash from customers
- counts, stores and stores it in high-security Cash-Centers
- distributes cash back out to different customers

Historically CashLog ran one Cash Center in every Spanish provincial capital, because the central bank had a branch there. That reason is now gone, and two pressures now push the other way:

- Cash is shrinking. Electronic payment keeps replacing physical cash, so the
  volume each center handles is falling.
- Cash Centers are expensive. Security, surveillance and insurance make every
  single building cost millions of euros per year just to keep open.

Today the network is **42 Cash Centers** serving about **42,000 customer
locations**. The 42,000 customers have already been grouped into*515 customer regions.

> **The decision:** Out of the 42 existing Cash Centers, which
> subset do we **keep open** and which do we **close**, so that the **total yearly
> cost is minimal** while **every one of the 515 regions is still served**?

**Two hard constraints from the business side:**

1. We do not open new locations. We only choose among the 42 that exist.

2. Every region must remain served. Closing a center is only allowed if its
   regions can be picked up by other centers.

## Formalizing the Problem

### Underlying Trade-off

We cannot just close the expensive centers, because there is an underlying trade-off that drives our decision problem:

If we open more centers, we will lower driving time/distance so we reduce transport cost, but have high fixed costs for running the Cash-Center
If we open fewer centers, we have low fixed costs but longer distances, so transport costs increase. 

The cheapest, most stable and most optimal network sits somewhere in the middle.

### Why optimizing the network as a whole matters

Scoring each Center individually and closing the worst one is not possible because of network effects. Wether a Center is worth keeping depends on which other centers are open. If one Center is closed, its regions have to be reasssigned to different Centers, which changes their loads and costs parameters. 

The Regions are interdependent, so we must decide all of them simultaneously by using a optimization model. 

### Starting point. Warehouse-location model

The classic template for "which facilities should we keep open" is the Warehouse-Location model. 

**The vocabulary**

| Symbol | Meaning |
|---|---|
| $i$ | a Cash Center (one of the 42), the "warehouse" |
| $j$ | a customer region (one of the 515) |
| $f_i$ | yearly fixed cost of running center $i$ |
| $c_{ij}$ | yearly transport cost of serving region $j$ from center $i$ |
| $y_i \in \{0,1\}$ | **decision:** is center $i$ open (1) or closed (0)? |
| $x_{ij} \in \{0,1\}$ | **decision:** is region $j$ served by center $i$? |

**The model:**

$$\min_{x,y}\; \underbrace{\sum_{i}\sum_{j} x_{ij}\,c_{ij}}_{\text{transport cost}}
\;+\; \underbrace{\sum_{i} f_i\,y_i}_{\text{fixed cost of open centers}}$$

subject to

$$\sum_{i} x_{ij} = 1 \;\;\forall j \quad\text{(every region served by exactly one center)}$$
$$x_{ij} \le y_i \;\;\forall i,j \quad\text{(you may only use a center that is open)}$$

This model is our starting point for the logic. But important core assumptions of this model do not apply for CashLog.

#### How the models breaks 

Three of the models build in assumptions cannot be applied to our CashLog Problem: 

1. $c_{ij}$ has no obvious value
The model assumes a known cost-per-link $c_{ij}$ exists. We do not have a "cost to serve region $j$ from center $i$" price tag in the raw data. Trucks
serve many customers per 8-hour shift on a route, not one round trip per
customer. We must therefore derive $c_{ij}$ from shift logic 

2. individual customers is the wrong unit of analysis
The model assumes a manageable, fixed set of "customers" $j$. Optimising location
choices against 42,000 individual points is computationally unpractical and not how real life routing logic works (Trucks don't make an isolated trip for a single customer). This is why customers need to be processed into cluster regions. 

3. Fixed cost $f_i$ is treated as one constant per center
The model assumes each facility has one fixed cost number, independent of how much
it processes. At CashLog, capacity is a strategic, investable choice. Centers can be build in different sizes. Cost follows economies of scale. A bigger center costs more in total but less per delivery. A single constant $f_i$ cannot represent that.

**Consequence:** points 1 and 2 are pre-processing problems we solve before the
model runs (Part 3 and the existing clustering); point 3 is a structural flaw in
the objective itself. Fixing it means replacing the single constant $f_i$ with a
cost structure that depends on how much volume a center actually handles. We will 
build that **extended model** later, once we have looked at the data closely
enough to calibrate it properly.


In [1]:
#shared imports
import folium
import numpy as np
import marimo as mo
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pulp as pl
import highspy  # noqa: F401  (backend for pl.HiGHS, same as base notebook)

## Data

We load four tables from the course's public GitHub repository.

In [2]:
_base = "https://raw.githubusercontent.com/D3IP-SS25/data-driven-scm-dataset/refs/heads/main/data"
warehouses = pd.read_csv(f"{_base}/warehouses.csv", index_col="warehouseID")
regions = pd.read_csv(f"{_base}/regions.csv", index_col="regionID")
shifts = pd.read_csv(f"{_base}/shifts.csv", index_col=["warehouseID", "regionID"])
shifts_ref = pd.read_csv(f"{_base}/shifts_with_costs.csv", index_col=["warehouseID", "regionID"])

In [3]:
#helper function for data analysis
from IPython.display import display, Markdown

def quick_look(df: pd.DataFrame, name: str, exclude: list[str] | None = None):
    missing = df.isna().sum()
    missing = missing[missing > 0]

    meta_md = f"**`{name}`** — {df.shape[0]:,} rows × {df.shape[1]} columns\n\n"
    meta_md += "| column | dtype | missing |\n|---|---|---|\n"
    meta_md += "\n".join(f"| `{c}` | {df[c].dtype} | {missing.get(c, 0)} |" for c in df.columns)

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    id_like = [c for c in numeric_cols if c.lower().endswith(("id", "_id"))]
    drop = set(id_like) | set(exclude or [])
    numeric_cols = [c for c in numeric_cols if c not in drop]

    display(Markdown(meta_md))
    display(df.head(3))
    if numeric_cols:
        display(df[numeric_cols].describe().round(2))
    else:
        display(Markdown("_no non-ID numeric columns to describe_"))

In [4]:
quick_look(warehouses, "warehouses")

**`warehouses`** — 42 rows × 4 columns

| column | dtype | missing |
|---|---|---|
| `city` | object | 0 |
| `fixedCosts` | int64 | 0 |
| `lat` | float64 | 0 |
| `lon` | float64 | 0 |

,city,fixedCosts,lat,lon
warehouseID,,,,
1,Vitoria,1344000,42.846509,-2.672403
2,Albacete,2580000,38.989012,-1.854870
3,Alicante,4572000,38.353738,-0.490185


,fixedCosts,lat,lon
count,42.00,42.00,42.00
mean,3394857.14,40.92,-3.41
std,5609205.28,1.94,2.74
min,1344000.00,36.84,-8.47
25%,1344000.00,39.57,-5.44
50%,2580000.00,41.49,-3.65
75%,2580000.00,42.39,-1.89
max,35904000.00,43.46,2.82


In [5]:
quick_look(regions, "regions")

**`regions`** — 515 rows × 6 columns

| column | dtype | missing |
|---|---|---|
| `zipCode` | int64 | 0 |
| `city` | object | 0 |
| `lat` | float64 | 0 |
| `lon` | float64 | 0 |
| `yearlyDemand` | int64 | 0 |
| `minutesPerStop` | float64 | 0 |

,zipCode,city,lat,lon,yearlyDemand,minutesPerStop
regionID,,,,,,
1,40297,SANCHONUÑO,41.3236,-4.3050,1511,15.761166
2,25220,BELL-LLOCH,41.6333,0.7833,2535,14.250221
3,45695,ALBERCHE DEL CAUDILLO,39.9189,-4.6801,679,12.666904


,zipCode,lat,lon,yearlyDemand,minutesPerStop
count,515.00,515.00,515.00,515.00,515.00
mean,25766.61,40.66,-3.68,5548.34,17.05
std,14860.36,1.93,2.84,18774.60,4.69
min,1009.00,36.55,-9.19,16.00,6.20
25%,13660.00,39.16,-5.76,573.50,13.92
50%,25003.00,40.95,-3.93,1212.00,16.32
75%,39931.50,42.34,-1.77,3362.00,19.74
max,50830.00,43.66,3.10,247638.00,32.12


In [6]:
quick_look(shifts, "shifts")

**`shifts`** — 21,630 rows × 2 columns

| column | dtype | missing |
|---|---|---|
| `transportationCosts` | float64 | 0 |
| `travelTime` | float64 | 0 |

transportationCosts  travelTime
warehouseID regionID                                 
1           74                 592.678040  592.678040
            268                692.621277  692.621277
            224                797.767822  797.767822

,transportationCosts,travelTime
count,21630.00,21630.00
mean,562.87,562.87
std,292.87,292.87
min,0.00,0.00
25%,347.26,347.26
50%,535.28,535.28
75%,754.16,754.16
max,1604.89,1604.89


In [7]:
shifts

transportationCosts  travelTime
warehouseID regionID                                 
1           74                 592.678040  592.678040
            268                692.621277  692.621277
            224                797.767822  797.767822
            481                 89.000000   89.000000
            478                263.676605  263.676605
...                                   ...         ...
50          253                713.992920  713.992920
            439                539.706177  539.706177
            393                451.236511  451.236511
            445                262.172546  262.172546
            170                448.638672  448.638672

[21630 rows x 2 columns]

We use three diferent datasets: 

- warehouses.csv
- regions.csv
- shifts_with_costs.csv

The warehouses dataset contains the ciy, location and fixed costs of opening/running each one of the 42 possible warehouses. The fixed costs range from 1.344.000,00 to 35.904.000,00, likely depending on the size and property price level of that certain region.

The regions dataset contains informations for each of the 515 clustered demand regions. The dataset has therefore already solved the second major problem of the base model we had to tackle. The regions were clustered by zipCode. For every region each row contains a unique regionID, the name of the city, its location, yearly Demand and average minutesPerStop. Yearly demand ranges from 16,00 to 247.638,00 and minutes per stop take anywhere from 6.20 to 32.12min. The minutesPerStop column also already bundels the driving between customers inside a region with the time spent at each customer. It is therefore a single pre-summerised productivity number.

The third shifts_with_costs dataset contains the travelTime in Minutes and trasportationCosts for every (center, region) pair, totaling 21,360 rows. Transportation costs in this dataset is just a placeholder. It equals the travel time. We will derive sensible and logic proxys for the travel time in the following step. 

## Estimating the transport cost $c_{ij}$

Our raw dataset only gives us the driving time as a placeholder for the yearly cost to serve region j from center i. We therefore need to derive an estimate using shift logic the business actually uses: 

A Truck leaves the center, drives to the region, works and drives back to the Cash-Center.

1. Working time per shift: The round trip uses $2 \cdot \text{travelTime}_{ij}$ minutes, (aber nur falls traveltime nur von center zu region ist und nicht wieder zurück, ist aber sehr wahrscheinlich) so the time available for actually serving customers is:

$$\text{usable time} = \underbrace{450}_{\text{shift}} - 2\cdot \text{travelTime}_{ij}.$$

this is the time we have left to actually do service, after considering the round trip. 

2. Ammount of stops per shift:
Each stop takes minutesPerStop (=driving between shops + time spent at the shop):

$$\text{stops per shift} = \frac{450 - 2\,\text{travelTime}_{ij}}{\text{minutesPerStop}_j}.$$

to calculate the average stops per shift we have to divide the usable time, by the average time per stop.

3. Ammount of shifts a region requires per year:
The region needs yearlyDemand stops in total: 

$$\text{shifts per year} = \frac{\text{yearlyDemand}_j}{\text{stops per shift}}.$$

by dividing the yearly stop demand by the ammount of stops possible during a single shift, we can derive the total ammount of shifts an individual region needs per year, for every customer to be served.

4. deriving the transport costs:
To get a fesible value for the yearly transportation costs, we simply need to multiply the ammount of shifts per year by the price of an individual shift:

$$\; c_{ij} = 480 \cdot
\frac{\text{yearlyDemand}_j \cdot \text{minutesPerStop}_j}{\,450 - 2\,\text{travelTime}_{ij}\,} \;$$

#### Interpretation:

The farther away a region (the bigger the travel time), the smaller the denominater becomes, which increases $c_{ij}$. The more spread-out the customers (the bigger minutesPerStop), the more expensive and lastly, the more demand, the bigger the numerator, the more cost. 

### Unreachable links

Routes, where $2\cdot \text{travelTime}_{ij} \ge 450$, are not possible. Travel time would exeed our usable time per shift. In our model, the denominater would become zero or negative.  (jetzt DECISION: BIG M, damit das einfach rausgeworfen wird, oder einfach von vornerein keinen Link bauen)